In [48]:
import warnings
warnings.filterwarnings('ignore')
import os
import numpy as np
import pandas as pd
from datetime import datetime
from time import time
from scipy.sparse import csr_matrix
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.manifold import TSNE
from sklearn.manifold import MDS
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif, mutual_info_classif, chi2
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from functools import partial
from sklearn.feature_selection import SelectFromModel

In [2]:
f = '%Y-%m-%d %H:%M:%S'
PATH_TO_DATA = ''

**Загружаем данные**

In [3]:
train_df = pd.read_csv(os.path.join(PATH_TO_DATA, 'train_sessions.csv'),
                       index_col='session_id')
test_df = pd.read_csv(os.path.join(PATH_TO_DATA, 'test_sessions.csv'),
                      index_col='session_id')
train_test_df = pd.concat([train_df, test_df])

**Получаем из датафрейма отсортированный список уникальных сайтов. Не забываем выкинуть из него 0. Этот список нам понадобиться исключительно для того, чтобы при создании разряженной матрицы определить индексы дополнительных характеристик**

In [4]:
col_arr = ['site%d' % i for i in range(1, 11)]
train_test_df_sites = train_test_df[col_arr].fillna(0).astype('int')

uniq_sites = pd.unique(train_test_df_sites.stack()).tolist()
uniq_sites.remove(0)
sorted_uniq_sites = sorted(uniq_sites)

**Рассчитываем дополнительные характеристики: продолжительность сессии, и кол-во уникальных сайтов в сессии**

In [5]:
def session_length(row):
    time_stamps = row[['time%d' % i for i in range(1, 11)]].dropna().values
    if len(time_stamps) < 2:
        return 0
    conv_arr = [datetime.strptime(e, f) for e in time_stamps]
    time_diff = max(conv_arr) - min(conv_arr)
    return time_diff.total_seconds()


def unique_sites(row):
    sites = row[['site%d' % i for i in range(1, 11)]].dropna().values
    return len(np.unique(sites))

In [6]:
start = time()

train_test_df['session_length'] = train_test_df.apply(session_length, axis=1)
train_test_df['unique_sites'] = train_test_df.apply(unique_sites, axis=1)

col_arr = ['site%d' % i for i in range(1, 11)]
col_arr.append('session_length')
col_arr.append('unique_sites')

print('Время выполнения {} секунд'.format(time() - start))

Время выполнения 358.3299527168274 секунд


**Создаем датафрейм, с которым далее будем работать**

In [7]:
train_test_df_sites = train_test_df[col_arr].fillna(0).astype('int')
train_test_df_sites.head()

,site1,site2,site3,site4,site5,site6,site7,site8,site9,site10,session_length,unique_sites
session_id,,,,,,,,,,,,
1,718,0,0,0,0,0,0,0,0,0,0,1
2,890,941,3847,941,942,3846,3847,3846,1516,1518,26,7
3,14769,39,14768,14769,37,39,14768,14768,14768,14768,7,4
4,782,782,782,782,782,782,782,782,782,782,270,1
5,22,177,175,178,177,178,175,177,177,178,246,4


**Формируем массивы для разряженной матрицы и саму разряженную матрицу**

In [8]:
V = []  # non zero values
COL_INDEX = []  # col indices of non zero values
ROW_INDEX = []  # row_start = ROW_INDEX[row] & row_end = ROW_INDEX[row + 1]

start = time()

X_val = train_test_df_sites.values
session_len_idx = max(sorted_uniq_sites)
uniq_sites_idx = max(sorted_uniq_sites) + 1

row_cnt = 0
ROW_INDEX.append(row_cnt)

for num_row in range(train_test_df_sites.shape[0]):
    row = X_val[num_row][:-2]
    num_occurrence = dict(zip(*np.unique(row, return_counts=True)))
    num_occurrence.pop(0, None)
    num_occurrence = dict(sorted(num_occurrence.items()))

    for k, v in num_occurrence.items():
        V.append(v)
        COL_INDEX.append(k - 1)

    
    session_len = X_val[num_row][10]
    if session_len > 0:
        row_cnt = row_cnt + 1
        V.append(session_len)
        COL_INDEX.append(session_len_idx)

    
    uniq_sites = X_val[num_row][11]
    if uniq_sites > 0:
       row_cnt = row_cnt + 1
       V.append(uniq_sites)
       COL_INDEX.append(uniq_sites_idx)

    row_cnt = row_cnt + len(num_occurrence)
    ROW_INDEX.append(row_cnt)
    

csr_X = csr_matrix((V, COL_INDEX, ROW_INDEX))
print('Время выполнения {} секунд'.format(time() - start))

Время выполнения 8.917265892028809 секунд


**Делим разряженную матрицу на обучающую и тестовую, отсекая две последние колонки, тк в них доп характеристики**

In [9]:
csr_X_train = csr_X[:len(train_df), :-2]
csr_X_test = csr_X[len(train_df):, :-2]
y = train_df['target'].values

**Отвечаем на <font color="red">Вопрос 1</font>**

In [10]:
print('{} {} {} {}'.format(csr_X_train.shape[0], csr_X_train.shape[1],
                          csr_X_test.shape[0], csr_X_test.shape[1]))

253561 48371 82797 48371


**Делим обучающую выборку на две части**

In [11]:
train_share = int(.7 * csr_X_train.shape[0])
X_train, y_train = csr_X_train[:train_share, :], y[:train_share]
X_valid, y_valid = csr_X_train[train_share:, :], y[train_share:]

**Обучаем классификатор и делаем предсказание**

In [12]:
sgd_logit = SGDClassifier(n_jobs=-1, random_state=17, loss='log_loss')
sgd_logit.fit(X_train, y_train)

y_pred = sgd_logit.predict_proba(X_valid)

**Считаем метрику roc_auc отвечаем на <font color="red">Вопрос 2</font>**

In [13]:
print('sgd roc_auc score = {}'.format(roc_auc_score(y_valid, y_pred.transpose()[1])))

sgd roc_auc score = 0.9336212361738347


**Заново обучаем классификатор уже на обучающей выборке целиком, строим предсказания**

In [14]:
clf = SGDClassifier(n_jobs=-1, random_state=17, loss='log_loss')
clf.fit(csr_X_train, y)

y_pred = clf.predict_proba(csr_X_test)

In [15]:
def write_to_submission_file(predicted_labels, out_file,
                             target='target', index_label="session_id"):
    # turn predictions into data frame and save as csv file
    predicted_df = pd.DataFrame(predicted_labels,
                                index=np.arange(1, predicted_labels.shape[0] + 1),
                                columns=[target])
    predicted_df.to_csv(out_file, index_label=index_label)

**Записываем предсказания в файл для отправки на kaggle**

In [16]:
out_csv = 'out_prob.csv'
write_to_submission_file(y_pred.transpose()[1], out_csv)

**Загружаем результаты на Kaggle (имя Nikolay Poluyaktov), получаем результат <font color="red">0.91646</font>, что больше чем sgd_logit_benchmark.csv. Если мы добавим в классификатор две характеристики: количество уникальных сайтов в сессии, и продолжительность сессии - это не улучшит работу классификатора. Рассчет тут полностью аналогичный, его оставляем за скобками.**

**Попробуем улучишть результат подбором параметров классификации**

In [17]:
clf = SGDClassifier(n_jobs=-1, random_state=17, loss='log_loss', penalty="elasticnet")

param_grid = {
    "average": [True, False],
    "l1_ratio": np.linspace(0, 1, num=10),
    "alpha": np.power(10, np.arange(-5, 1, dtype=float))
}
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=17)

grid_search = GridSearchCV(clf, param_grid=param_grid, cv=skf, scoring='roc_auc')

start = time()
grid_search.fit(csr_X_train, y)
print(
    "GridSearchCV took %.2f seconds for %d candidate parameter settings."
    % (time() - start, len(grid_search.cv_results_["params"]))
)


def report(results, n_top=3):
    for i in range(1, n_top + 1):
        candidates = np.flatnonzero(results["rank_test_score"] == i)
        for candidate in candidates:
            print("Model with rank: {0}".format(i))
            print(
                "Mean validation score: {0:.3f} (std: {1:.3f})".format(
                    results["mean_test_score"][candidate],
                    results["std_test_score"][candidate],
                )
            )
            print("Parameters: {0}".format(results["params"][candidate]))
            print("")


report(grid_search.cv_results_)

GridSearchCV took 193.24 seconds for 120 candidate parameter settings.
Model with rank: 1
Mean validation score: 0.954 (std: 0.003)
Parameters: {'alpha': 9.999999999999999e-06, 'average': False, 'l1_ratio': 0.0}

Model with rank: 2
Mean validation score: 0.953 (std: 0.003)
Parameters: {'alpha': 9.999999999999999e-06, 'average': False, 'l1_ratio': 0.1111111111111111}

Model with rank: 3
Mean validation score: 0.952 (std: 0.003)
Parameters: {'alpha': 9.999999999999999e-06, 'average': False, 'l1_ratio': 0.2222222222222222}



**Оптимальными параметрами являются параметры по умолчанию**

In [46]:
selector = SelectKBest()
sgd_logit = SGDClassifier(n_jobs=-1, random_state=17, loss='log_loss', penalty="elasticnet")
pipe = Pipeline(steps=[("selector", selector), ("logistic", sgd_logit)])

In [ ]:
param_grid = {
    "selector__k": [6000, 7000, 8000],
    "selector__score_func": [partial(f_classif), partial(chi2)],
    "logistic__average": [True, False],
    "logistic__l1_ratio": np.linspace(0, 1, num=10),
    "logistic__alpha": np.power(10, np.arange(-5, 1, dtype=float))
}
search = GridSearchCV(pipe, param_grid, n_jobs=2, scoring='roc_auc')
search.fit(csr_X_train, y)
print("Best parameter (CV score=%0.5f):" % search.best_score_)
print(search.best_params_)

/opt/conda/envs/polyanka/lib/python3.9/site-packages/sklearn/feature_selection/_univariate_selection.py:112: UserWarning: Features [0 0 0 ... 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/opt/conda/envs/polyanka/lib/python3.9/site-packages/sklearn/feature_selection/_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/opt/conda/envs/polyanka/lib/python3.9/site-packages/sklearn/feature_selection/_univariate_selection.py:112: UserWarning: Features [0 0 0 ... 0 0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/opt/conda/envs/polyanka/lib/python3.9/site-packages/sklearn/feature_selection/_univariate_selection.py:113: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
/opt/conda/envs/polyanka/lib/python3.9/site-packages/sklearn/feature_selection/_univariate_selection.py:112: UserWarning: Features [0 0 0 ... 0 0 0] are con